In [ ]:
import string
import warnings
warnings.filterwarnings('ignore')

try:
    import nltk
    from nltk.tokenize import word_tokenize
    from nltk.corpus import stopwords
    
    try:
        nltk.download('punkt_tab', quiet=True)
        nltk.download('stopwords', quiet=True)
    except:
        nltk.download('punkt', quiet=True)
        nltk.download('stopwords', quiet=True)
        
except ImportError:
    print("Установите NLTK: pip install nltk")
    import sys
    sys.exit(1)

<p>Работа с текстом (базовая очистка)</p>

In [28]:
text = "Машинное обучение - это круто! А вы так не думаете?"

text_lower = text.lower()

text_without_punct = text_lower.translate(str.maketrans('', '', string.punctuation))

<p>Сейчас наше сообщение выглядит так: машинное обучение это круто а вы так не думаете</p>

<p>Токенизация</p>

In [29]:
try:
    tokens = word_tokenize(text_without_punct, language='russian')
    print(f'Токены: {tokens}')
except LookupError as e:
    print('Ошибка: не загружен ресурс для токенизации. Пробуем английский...')
    try:
        tokens = word_tokenize(text_without_punct, language='english')
        print(f'Токены (eng): {tokens}')
    except:
        tokens = text_without_punct.split()
        print(f'Токены (arm): {tokens}')

Токены: ['машинное', 'обучение', 'это', 'круто', 'а', 'вы', 'так', 'не', 'думаете']


<p>Удаление стоп-слов</p>

In [30]:
try:
    russian_stopwords = set(stopwords.words('russian'))
except LookupError as e:
    try:
        russian_stopwords = set(stopwords.words('english'))
    except:
        russian_stopwords = {'это', 'а', 'так', 'и', 'в', 'на', 'с', 'по', 'типо', 'но'}
print(f"Пример стоп-слов: {list(russian_stopwords)[:5]}...")

Пример стоп-слов: ['во', 'где', 'то', 'вам', 'была']...


In [31]:
filtered_tokens = [word for word in tokens if word not in russian_stopwords]
print(f"Токены после удаления стоп-слова: {filtered_tokens}")

Токены после удаления стоп-слова: ['машинное', 'обучение', 'это', 'круто', 'думаете']


<p>Лемматизация - процесс приведения слова к его базовой, начальной форме (лемме).</p>
<p>Лемма - это форма слова, которая представляет его в словаре и имеет обшие значения с иными формами.</p>

In [32]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()
lemmatized_tokens = []

for word in filtered_tokens:
    try:
        lemma = morph.parse(word)[0].normal_form
        lemmatized_tokens.append(lemma)
    except:
        lemmatized_tokens.append(word)

print(f"Лемматизированные токены: {lemmatized_tokens}")

Лемматизированные токены: ['машинное', 'обучение', 'это', 'круто', 'думаете']


<p>Векторизация текста</p>

In [33]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

documents = [
    'машинное обучение это круто',
    'обучение с учителем и обучение без учителя',
    'крутые курсы по машинному обучению'
]

In [34]:
for i, doc in enumerate(documents, 1):
    print(f'{i}. {doc}')

1. машинное обучение это круто
2. обучение с учителем и обучение без учителя
3. крутые курсы по машинному обучению


<p>CountVectorizer</p>

In [35]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

print(f'Признаки (слова): {vectorizer.get_feature_names_out()}')
print(f'Матрица признаков:')
print(f'{X.toarray()}')

Признаки (слова): ['без' 'круто' 'крутые' 'курсы' 'машинное' 'машинному' 'обучение'
 'обучению' 'по' 'учителем' 'учителя' 'это']
Матрица признаков:
[[0 1 0 0 1 0 1 0 0 0 0 1]
 [1 0 0 0 0 0 2 0 0 1 1 0]
 [0 0 1 1 0 1 0 1 1 0 0 0]]


<p>TF-IDF</p>

In [36]:
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(documents)

print(f'Признаки: {tfidf_vectorizer.get_feature_names_out()}')
print(f'TF-IDF матрица (округлено до 3 знаков):')
print(f'{X_tfidf.toarray().round(3)}')

Признаки: ['без' 'круто' 'крутые' 'курсы' 'машинное' 'машинному' 'обучение'
 'обучению' 'по' 'учителем' 'учителя' 'это']
TF-IDF матрица (округлено до 3 знаков):
[[0.    0.529 0.    0.    0.529 0.    0.402 0.    0.    0.    0.    0.529]
 [0.434 0.    0.    0.    0.    0.    0.66  0.    0.    0.434 0.434 0.   ]
 [0.    0.    0.447 0.447 0.    0.447 0.    0.447 0.447 0.    0.    0.   ]]


<p>
Интерпретация
<ul>
    <li>В BoW просто подсчет частоты слов.</li>
    <li>В TF-IDF веса отражают важность слов в контексте всех документов.</li>
</ul>
ПОЛНЫЙ ПАЙПЛАЙН ОБРАБОТКИ ТЕКСТА
</p>

In [37]:
import pandas as pd

reviews = [
    "Этот фильм просто отличный! Очень понравился.",
    "Скучный и нудный фильм, не советую.",
    "Ну такое себе, среднячок. Можно один раз посмотреть."
]

print("Исходные отзывы:")
for i, rev in enumerate(reviews, 1):
    print(f"{i}: {rev}")

Исходные отзывы:
1: Этот фильм просто отличный! Очень понравился.
2: Скучный и нудный фильм, не советую.
3: Ну такое себе, среднячок. Можно один раз посмотреть.


In [38]:
def preprocess_text(text, morph_analyzer, stop_words):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    processed = []
    for token in tokens:
        if token not in stop_words and len(token) > 2:
            try:
                lemma = morph_analyzer.parse(token)[0].normal_form
                processed.append(lemma)
            except:
                processed.append(token)
    return ' '.join(processed)

In [39]:
custom_stopwords = set(list(russian_stopwords) + ['этo', 'очень', 'можно'])

processed_reviews = []
for review in reviews:
    processed = preprocess_text(review, morph, custom_stopwords)
    processed_reviews.append(processed)

<p>Отзывы после предобработки:</p>

In [40]:
for i, rev in enumerate(processed_reviews, 1):
    print(f"{i}: {rev}")

1: фильм просто отличный понравиться
2: скучный нудный фильм советовать
3: такой среднячок посмотреть


In [41]:
# TF-IDF векторизация
tfidf = TfidfVectorizer()
X_final = tfidf.fit_transform(processed_reviews)

df = pd.DataFrame(
    X_final.toarray().round(3),
    columns=tfidf.get_feature_names_out()
)

df['Исходный_текст'] = reviews

print("Финальная матрица признаков (TF-IDF):")
print(df)

Финальная матрица признаков (TF-IDF):
   нудный  отличный  понравиться  посмотреть  просто  скучный  советовать  \
0   0.000     0.529        0.529       0.000   0.529    0.000       0.000   
1   0.529     0.000        0.000       0.000   0.000    0.529       0.529   
2   0.000     0.000        0.000       0.577   0.000    0.000       0.000   

   среднячок  такой  фильм                                     Исходный_текст  
0      0.000  0.000  0.402      Этот фильм просто отличный! Очень понравился.  
1      0.000  0.000  0.402                Скучный и нудный фильм, не советую.  
2      0.577  0.577  0.000  Ну такое себе, среднячок. Можно один раз посмо...  
